# Anomaly Detection Evaluation

## 1. Setup & Configuration

In [1]:
# Auto-reload modules
%load_ext autoreload
%autoreload 2

In [2]:
# Imports - Clean refactored API
import matplotlib.pyplot as plt
from pathlib import Path

from evaluation.runner import run_evaluation, run_single_object, run_few_shot, ExperimentResults, MethodResults
from evaluation.visualization import  plot_single_result, plot_method_comparison, plot_object_heatmap, plot_shot_count_analysis
from evaluation.export import  export_all


from few_shot.DINOv3Wrapper import DINOv3Wrapper
from mvtechDataset import ensure_mvtech_dataset_is_downloaded, get_categories, load_mvtec_data

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

✓ All imports successful


In [3]:
# Configuration
CONFIG = {
    'model_name': 'facebook/dinov3-vits16-pretrain-lvd1689m',
    'smaller_edge_size': 640,
    'k_values': [1, 2, 4, 8],
    'objects': None,  # None = all, or ['bottle', 'hazelnut']
    'use_rotation': False,
    'use_masking': False,
    'k_neighbors': 3,
    'output_dir': Path('evaluation_results_clean')
}

CONFIG['output_dir'].mkdir(exist_ok=True)
print('Configuration:', CONFIG)

Configuration: {'model_name': 'facebook/dinov3-vits16-pretrain-lvd1689m', 'smaller_edge_size': 640, 'k_values': [1, 2, 4, 8], 'objects': None, 'use_rotation': False, 'use_masking': False, 'k_neighbors': 3, 'output_dir': PosixPath('evaluation_results_clean')}


In [4]:
# Load dataset and model
dataset_path = ensure_mvtech_dataset_is_downloaded('.')
categories = get_categories(dataset_path)
objects_to_eval = CONFIG['objects'] if CONFIG['objects'] else categories

print(f'Found {len(categories)} categories')
print(f'Will evaluate: {objects_to_eval}')

model = DINOv3Wrapper(
    model_name=CONFIG['model_name'],
    smaller_edge_size=CONFIG['smaller_edge_size']
)

Dataset already exists at ./mvtec_anomaly_detection
Found 15 categories
Will evaluate: ['transistor', 'wood', 'capsule', 'metal_nut', 'pill', 'zipper', 'tile', 'hazelnut', 'carpet', 'cable', 'leather', 'screw', 'toothbrush', 'bottle', 'grid']
Loading facebook/dinov3-vits16-pretrain-lvd1689m on cpu...
Model loaded! Hidden dim: 384, Register tokens: 4, Patch size: 16
Smaller edge size: 640


## 2. Quick Single-Object Test

In [5]:
# Quick test
test_result = run_single_object(
    method_fn=run_few_shot,
    model=model,
    dataset_path=dataset_path,
    object_name='hazelnut',
    k=8,
    method_kwargs={'use_rotation': CONFIG['use_rotation'], 'use_masking': CONFIG['use_masking'], 'k_neighbors': CONFIG['k_neighbors']}
)

print('Metrics:')
for m, v in test_result.metrics.items():
    print(f'  {m}: {v:.4f}')

Loaded 8 reference images for hazelnut
Loaded 110 test images (70 anomalous, 40 normal)
Running 8_shot evaluation on hazelnut...
Building memory bank from 8 images...
Rotation augmentation: False


Processing reference images:   0%|          | 0/8 [00:00<?, ?it/s]

Memory bank built! Index size: 12800


Detecting anomalies (hazelnut):   0%|          | 0/110 [00:00<?, ?it/s]

  AUROC: 0.9986, AU-PRO: 0.0062, IoU: 0.1802
Metrics:
  auroc: 0.9986
  aupro: 0.0062
  pixel_auroc: 0.9930
  pixel_f1: 0.8427
  mean_iou: 0.1802


In [6]:
test_result

TypeError: plot_single_result() missing 1 required positional argument: 'anomaly_map'

## 3. Run Complete Evaluation

In [ ]:
# Run evaluation
results = run_evaluation(
    method_fn=run_few_shot,
    method_name='few_shot',
    model=model,
    dataset_path=dataset_path,
    objects=objects_to_eval,
    k_values=CONFIG['k_values'],
    method_kwargs={'use_rotation': CONFIG['use_rotation'], 'use_masking': CONFIG['use_masking'], 'k_neighbors': CONFIG['k_neighbors']}
)

results.description = f"Few-shot: rotation={CONFIG['use_rotation']}, masking={CONFIG['use_masking']}"
results.save(CONFIG['output_dir'] / 'experiment_results')
print('✓ Evaluation complete and saved')

## 4. Analyze Results

In [ ]:
# Summary
summary = results.summary_table()
print('Summary Statistics:')
print(summary.to_string(index=False))

best = summary.iloc[0]
print(f"\nBest: {best['Method']} - AUROC: {best['Mean AUROC']:.4f}")

In [ ]:
# Per-object
per_obj = results.per_object_table()
pivot = per_obj.pivot_table(index='Object', columns='Method', values='AUROC')
print('AUROC by Object:')
print(pivot.to_string())

## 5. Visualizations

In [ ]:
plot_method_comparison(results, metrics=['auroc', 'aupro'])

In [ ]:
plot_object_heatmap(results, metrics=['AUROC', 'AU-PRO'])

In [ ]:
if len(CONFIG['k_values']) > 1:
    plot_shot_count_analysis(results, k_values=CONFIG['k_values'], metrics=['auroc', 'aupro'])

## 6. Compare Methods

In [ ]:
if len(results.list_methods()) >= 2:
    comp = results.compare_methods('1_shot', '8_shot', metric='auroc')
    print(comp.to_string(index=False))

## 7. Export Results

In [ ]:
export_all(results, CONFIG['output_dir'] / 'exports')
print('✓ Results exported')

## 8. Add Your Own Method

See `evaluation/README.md` for complete instructions on adding custom methods.

**Quick template:**

```python
def run_my_method(model, reference_images, test_images, test_labels, test_masks, object_name, **kwargs):
    from evaluation import MethodResults, compute_auroc, compute_pro, compute_iou
    
    # Your detection logic
    scores = ...
    maps = ...
    
    # Compute metrics
    auroc = compute_auroc(test_labels, scores)
    # ... other metrics ...
    
    return MethodResults(
        method_name='my_method',
        object_name=object_name,
        scores=scores,
        maps=maps,
        metrics={'auroc': auroc, ...},
        config=kwargs
    )

# Use it
my_results = run_evaluation(method_fn=run_my_method, ...)
```